In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt
from src.postprocess_ephemsim import (
    load_datasize_and_fit_data,
    plot_datasize_and_fit_data,
    plot_bits,
    plot_ephemsize_M,
    test_fitting_L2,
    plot_fit_results_M,
)

In [ ]:
import os

basedir = "/Users/keidaiiiyama/Documents/sw_navlab/LuPNT-private/output/Ephemeris/"
esim_dir = basedir + "data/ephemeris/"
figdirbase = basedir + "figures/fitting/"
if not os.path.exists(figdirbase):
    os.makedirs(figdirbase)

In [ ]:
# orbit = "LCRNS"  # Choose from "ELFO", "Polar", "NRHO"
orbit = "Moonlight"
# orbit = "LCRNS"
# orbit = "LNSS"
# orbit = "Polar"

plot_type = "use_kep_fouriers"
# plot_type = "use_cheby_samplings"

figdir = figdirbase + f"{orbit}/"
if not os.path.exists(figdir):
    os.makedirs(figdir)

if orbit == "LCRNS":
    sma = 11315.94e3  #
elif orbit == "LNSS":
    sma = 6541.40e3
elif orbit == "Polar":
    sma = 3870.00e3
elif orbit == "Moonlight":
    sma = 9748.1e3

T = 2 * np.pi * np.sqrt(sma**3 / pnt.GM_MOON)

print("Orbit: {}".format(orbit))
print("Semi-major axis: {:.2f} km".format(sma / 1e3))
print("Orbital period: {:.2f} hours".format(T / 3600))

if plot_type == "use_kep_fouriers":
    use_kep_fouriers = [[True, True], [True, False], [False, False]]
    poly_types = ["chebyshev"]
    use_cheby_samplings = [True]
elif plot_type == "use_cheby_samplings":
    use_kep_fouriers = [[True, True]]
    poly_types = ["chebyshev"]
    use_cheby_samplings = [False, True]
else:
    raise ValueError("Invalid plot_type: {}".format(plot_type))

# use_kep_fouriers = [[True, False]]
# poly_types = ["chebyshev", "legendre", "monomial"]
if orbit == "Polar":
    orders_max = 30
else:
    orders_max = 20

In [ ]:
data = load_datasize_and_fit_data(
    orbit,
    use_meq=False,
    fit_mins=[60, 120, 240, 360, 480],
    poly_types=poly_types,
    use_kep_fouriers=use_kep_fouriers,
    use_cheby_samplings=use_cheby_samplings,
    esim_dir=esim_dir,
    orders_max=orders_max,
)

### plot datasize vs ephemeris size

In [ ]:
if plot_type == "use_kep_fouriers":
    ylim = (1e-4, 1e4)
    percentile = "95"
    use_log = True
    labels = ["Cheby + OE + Fourier", "Cheby + OE", "Cheby"]
else:
    ylim = (1e-3, 1e3)
    percentile = "99.7"
    use_log = True
    labels = ["Uniform", "Chebyshev"]

if orbit == "LCRNS":
    pos_limit = 3.0  # 13.43*2/3*1/3
    vel_limit = 0.25  # 1.2*2/3*1/3
    print("LCRNS limits:", pos_limit, vel_limit)
    plot_datasize_and_fit_data(
        data,
        pos_limit=pos_limit,
        vel_limit=vel_limit,
        percentile=percentile,
        use_log=use_log,
        ylim=ylim,
        figname=figdir + f"/datasize_and_fit_{plot_type}.pdf",
        plot_legends=False,
        labels=labels,
    )
else:
    plot_datasize_and_fit_data(
        data,
        pos_limit=10.0,
        vel_limit=2.5,
        percentile=percentile,
        use_log=use_log,
        ylim=ylim,
        figname=figdir + f"/datasize_and_fit_{plot_type}.pdf",
        plot_legends=False,
        labels=labels,
    )

### Plot the number of bits per ephemeris parameter

In [ ]:
plot_bits(
    data,
    fit_min=360,
    bit_only=True,
    use_max_order=False,
    use_order=None,
    add_title=True,
    figdir=figdir,
)

### Plot the changes in the coefficients per mean anomaly

In [ ]:
fit_min = 480

for use_fourier in [False, True]:
    config = {
        "order": 12,
        "use_kep": True,
        "use_rsw": False,
        "use_fourier": use_fourier,
        "use_meq": False,
        "poly_type": "chebyshev",  # 'legendre', 'monomial', 'chebyshev'
        "sampling_type": "cheby",
    }

    plot_ephemsize_M(orbit, config, fit_min=fit_min, basedir=basedir)

### Plot Mean Anomaly vs Fitting Error

In [ ]:
fit_min = 360
order = 10

config_plot = {
    "poly_type": "chebyshev",
    "use_kep": True,
    "use_fourier": True,
    "use_cheby_sampling": True,
}

figname = (
    basedir + f"figures/M_errors/fit_results_M_{orbit}_fit{fit_min}_order{order}.pdf"
)

plot_fit_results_M(data, fit_min, order, config_plot, orbit, figname=figname)

### Test Different Weights for WLS vs Message Size Tradeoff

In [ ]:
# save the results to file
import pickle
import os

# orbit = "LCRNS"  # Choose from "ELFO", "Polar", "NRHO"
if orbit == "LCRNS" or orbit == "NRHO":
    fit_min = 480
    orders = [8, 9, 10, 11, 12, 13, 14, 15]
    use_kep = True
    use_fourier = True
elif orbit == "LNSS":
    fit_min = 360
    orders = [8, 9, 10, 11, 12]
    use_kep = True
    use_fourier = True
elif orbit == "Moonlight":
    fit_min = 360
    orders = [6, 7, 8, 9, 10, 11, 12]
    use_kep = True
    use_fourier = True
else:
    fit_min = 360
    orders = [11, 12, 13, 14, 15, 16]
    use_kep = True
    use_fourier = False

savedir = basedir + "data/ephemeris/l2_fitting_results/{}/kep_{}_fourier_{}/".format(
    orbit, use_kep, use_fourier
)
if not os.path.exists(savedir):
    os.makedirs(savedir)

config = {
    "order": 10,
    "use_kep": use_kep,
    "use_rsw": False,
    "use_fourier": use_fourier,
    "use_meq": False,
    "poly_type": "chebyshev",  # 'legendre', 'monomial', 'chebyshev'
    "sampling_type": "cheby",
}

alphas = np.power(10, np.linspace(-5, -2, 10))  # L2 regularization parameters

esims = []

for order in orders:
    print("========================================")
    print("Fitting with order = {}".format(order))
    print("========================================")
    config["order"] = order
    print("Order: {}".format(order))

    filename = savedir + "fitres_fitmin{}_order_{}.pkl".format(fit_min, order)

    if os.path.exists(filename):
        print("File {} already exists. Loading existing results...".format(filename))
        with open(filename, "rb") as f:
            esim_data, alphas, fit_min = pickle.load(f)
        print("Loaded result from file.")
    else:
        esim, eph = test_fitting_L2(
            orbit, config, fit_min, alphas=alphas, basedir=basedir, fit_obj="l1-log"
        )

        esim_data = {
            "fit_results": esim.fit_results,
            "datasizes": esim.datasizes,
            "M_array": esim.M_array,
            "orbdata": esim.orbdata,
            "config": config,
            "orbit": orbit,
            "basedir": basedir,
        }

        # save to file
        with open(filename, "wb") as f:
            pickle.dump((esim_data, alphas, fit_min), f)
        print("Results saved to {}".format(filename))

    # append to lists
    esims.append(esim_data)

In [ ]:
from src.postprocess_ephemsim import plot_fitting_l2_results

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# plot range
minbit = 700
maxbit = 1100
minpos = 0
maxpos = 12

config_str = ""
if use_kep:
    config_str += "coe"
config_str += " + chebyshev"
if use_fourier:
    config_str += " + fourier"
config_str += " (sampling:chebyshev)"
print("Config str: {}".format(config_str))

for i, esim_data in enumerate(esims):
    order = orders[i]
    print("Plotting results for order = {}".format(order))
    plot_fitting_l2_results(
        fig,
        axes,
        esim_data,
        fit_min,
        alphas,
        plotlabel="order={}".format(order),
        plot_thresholds=True,
    )

# plot the p95 position error vs total bits from data
configs = data[fit_min].keys()
for k, ephem_config in enumerate(configs):
    if config_str in ephem_config:
        print("Plotting data for config: {}".format(ephem_config))
        orders_plot = data[fit_min][ephem_config]["orders"]
        # plot_idx = orders_plot <= max(orders)
        # orders_plot = orders_plot[plot_idx]
        pos_p95 = data[fit_min][ephem_config]["diff_pos_p95"]
        total_bits = data[fit_min][ephem_config]["datasizes"]
        axes[2].plot(total_bits, pos_p95, "ks--", label="weight=0")
        for i, order_plot in enumerate(orders_plot):
            if (
                total_bits[i] > minbit
                and total_bits[i] < maxbit
                and pos_p95[i] < maxpos
                and pos_p95[i] > minpos
            ):
                axes[2].text(
                    total_bits[i] + 5, pos_p95[i], str(int(order_plot)), fontsize=10
                )

for ax in axes:
    ax.legend(fontsize=10)

axes[2].set_ylim(0, 12)
axes[2].set_xlim(650, 1100)
plt.tight_layout()
figdir = basedir + "figures/l2_fitting_results/"
if not os.path.exists(figdir):
    os.makedirs(figdir)
plt.savefig(
    figdir + "l2_fitting_results_{}_fitmin_{}.pdf".format(orbit, fit_min), dpi=300
)
plt.show()